# CEG-WM Paper Ablation CUDA Notebook

本 Notebook 面向 Google Colab 的论文级单对象单变量消融。

核心语义：
- 单次 base embed；
- 多次 detect rerun；
- 所有 variant 复用同一份 embed_record；
- 输出自动按 variant 后缀区分；
- 汇总 compare summary / compare table 便于论文制表与审计。

本 Notebook 只负责环境准备、参数定义、调用 scripts/run_paper_ablation_workflow.py 与结果展示。

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import psutil

try:
    from google.colab import drive
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORK_DIR = Path('/content') if IN_COLAB else Path.cwd()
TARGET_DIR = WORK_DIR / 'CEG-WM'
WORK_DIR.mkdir(parents=True, exist_ok=True)

def run_checked(command, cwd=None):
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        capture_output=True,
        text=True,
        encoding='utf-8',
        errors='replace',
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f'command failed: {command}')
    return result

print('=' * 80)
print('初始化 Paper Ablation Colab 环境')
print('=' * 80)
print(f'IN_COLAB={IN_COLAB}')
print(f'WORK_DIR={WORK_DIR}')
print(f'Python={sys.version.split()[0]}')
print(f'CPU cores={psutil.cpu_count()}')
print(f'RAM total={psutil.virtual_memory().total / (1024 ** 3):.1f} GB')
print(f'RAM available={psutil.virtual_memory().available / (1024 ** 3):.1f} GB')
gpu_probe = subprocess.run(['nvidia-smi'], capture_output=True, text=True, encoding='utf-8', errors='replace')
if gpu_probe.returncode == 0:
    print(gpu_probe.stdout)
else:
    print('nvidia-smi 不可用，当前可能不是 GPU Runtime。')

## 第 1 步：挂载 Google Drive

建议把结果同步到 Google Drive，便于保留 run_root、compare summary 与 CSV。

In [ ]:
USE_GOOGLE_DRIVE = True
GOOGLE_DRIVE_ROOT = None
GOOGLE_DRIVE_EXPORT_DIR = None

print('=' * 80)
print('挂载 Google Drive')
print('=' * 80)

if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount('/content/drive', force_remount=False)
    GOOGLE_DRIVE_ROOT = Path('/content/drive/MyDrive')
    GOOGLE_DRIVE_EXPORT_DIR = GOOGLE_DRIVE_ROOT / 'CEG_WM_Outputs' / 'Paper_Ablation_Cuda'
    GOOGLE_DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'✓ Google Drive 已挂载：{GOOGLE_DRIVE_ROOT}')
    print(f'✓ 导出目录：{GOOGLE_DRIVE_EXPORT_DIR}')
else:
    print('ℹ 跳过 Google Drive 挂载')

## 可编辑 Cell 1：基础参数与输出目录

这里定义仓库、配置、输出目录、是否安装依赖、是否执行可选 calibrate / evaluate。

默认建议：
- INSTALL_REQUIREMENTS = False；
- ENABLE_CALIBRATION = False；
- ENABLE_EVALUATE = False。

In [ ]:
REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
REPO_BRANCH = 'main'
CONFIG_PATH = 'configs/ablation/paper_ablation_cuda.yaml'
OUTPUT_ROOT = 'outputs/Paper_Ablation_Cuda'
RUN_TAG = 'paper_ablation_demo'
INSTALL_REQUIREMENTS = False
DOWNLOAD_MODELS = True
ENABLE_CALIBRATION = False
ENABLE_EVALUATE = False
HF_TOKEN = os.environ.get('HF_TOKEN', '')

print({
    'CONFIG_PATH': CONFIG_PATH,
    'OUTPUT_ROOT': OUTPUT_ROOT,
    'RUN_TAG': RUN_TAG,
    'INSTALL_REQUIREMENTS': INSTALL_REQUIREMENTS,
    'DOWNLOAD_MODELS': DOWNLOAD_MODELS,
    'ENABLE_CALIBRATION': ENABLE_CALIBRATION,
    'ENABLE_EVALUATE': ENABLE_EVALUATE,
})

## 可编辑 Cell 2：variant 列表

这里仅选择本次要跑的 variant 名称；完整定义仍来自 configs/ablation/paper_ablation_cuda.yaml。

In [ ]:
SELECTED_VARIANTS = [
    'GEO-on',
    'GEO-off',
]

print('本次 variant 列表：')
for variant_name in SELECTED_VARIANTS:
    print(f'  - {variant_name}')

## 第 2 步：同步仓库、安装依赖、验证关键模块导入

这一阶段会：
- 同步仓库；
- 可选执行 requirements 安装；
- 验证 run_embed / run_detect / run_paper_ablation_workflow 可导入。

In [ ]:
if TARGET_DIR.exists() and (TARGET_DIR / '.git').exists():
    run_checked(['git', '-C', str(TARGET_DIR), 'fetch', 'origin', REPO_BRANCH, '--depth', '1'])
    run_checked(['git', '-C', str(TARGET_DIR), 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'])
    run_checked(['git', '-C', str(TARGET_DIR), 'reset', '--hard', f'origin/{REPO_BRANCH}'])
else:
    if TARGET_DIR.exists():
        shutil.rmtree(TARGET_DIR)
    run_checked(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(TARGET_DIR)])

if INSTALL_REQUIREMENTS:
    run_checked([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=TARGET_DIR)

if str(TARGET_DIR) not in sys.path:
    sys.path.insert(0, str(TARGET_DIR))

print('=' * 80)
print('验证关键模块导入')
print('=' * 80)
from main.cli import run_embed as _run_embed_module
from main.cli import run_detect as _run_detect_module
from scripts import run_paper_ablation_workflow as _ablation_workflow_module
print('✓ main.cli.run_embed 导入成功')
print('✓ main.cli.run_detect 导入成功')
print('✓ scripts.run_paper_ablation_workflow 导入成功')

## 第 3 步：下载模型权重与 InSPyReNet

这一阶段检查并准备：
- SD3 相关 Hugging Face 模型缓存；
- InSPyReNet 权重 models/inspyrenet/ckpt_base.pth。

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download

CEG_WM_ROOT = TARGET_DIR
MODEL_CACHE_DIR = CEG_WM_ROOT / 'models' / 'hf_cache' / 'sd35_medium'
INSPYRENET_DIR = CEG_WM_ROOT / 'models' / 'inspyrenet'
INSPYRENET_DIR.mkdir(parents=True, exist_ok=True)
INSPYRENET_WEIGHT_PATH = INSPYRENET_DIR / 'ckpt_base.pth'

print('=' * 80)
print('下载模型权重与 InSPyReNet')
print('=' * 80)

if DOWNLOAD_MODELS:
    snapshot_kwargs = {
        'repo_id': 'stabilityai/stable-diffusion-3.5-medium',
        'local_dir': str(MODEL_CACHE_DIR),
        'local_dir_use_symlinks': False,
        'resume_download': True,
    }
    if HF_TOKEN:
        snapshot_kwargs['token'] = HF_TOKEN
    try:
        snapshot_download(**snapshot_kwargs)
        print(f'✓ SD3.5 Medium 已缓存到：{MODEL_CACHE_DIR}')
    except Exception as exc:
        print(f'⚠ SD3.5 Medium 下载失败：{type(exc).__name__}: {exc}')
        print('  若模型已在 Colab/HF cache 中，可继续；否则后续运行会失败。')

    if not INSPYRENET_WEIGHT_PATH.exists():
        inspyrenet_candidates = [
            ('RICHAAARC/InSPyReNet-Weights', 'ckpt_base.pth'),
            ('jwwangchn/InSPyReNet', 'ckpt_base.pth'),
        ]
        downloaded = False
        for repo_id, filename in inspyrenet_candidates:
            try:
                download_kwargs = {
                    'repo_id': repo_id,
                    'filename': filename,
                    'local_dir': str(INSPYRENET_DIR),
                    'local_dir_use_symlinks': False,
                }
                if HF_TOKEN:
                    download_kwargs['token'] = HF_TOKEN
                hf_hub_download(**download_kwargs)
                downloaded = True
                print(f'✓ InSPyReNet 权重已下载：{repo_id}/{filename}')
                break
            except Exception as exc:
                print(f'⚠ 候选 InSPyReNet 权重下载失败：{repo_id}/{filename} -> {type(exc).__name__}: {exc}')
        if not downloaded:
            print('⚠ 未能自动下载 InSPyReNet 权重，请手工放置到 models/inspyrenet/ckpt_base.pth')
    else:
        print(f'✓ InSPyReNet 权重已存在：{INSPYRENET_WEIGHT_PATH}')
else:
    print('ℹ 已跳过模型下载')

print(f'SD3 缓存目录存在：{MODEL_CACHE_DIR.exists()}')
print(f'InSPyReNet 权重存在：{INSPYRENET_WEIGHT_PATH.exists()}')

## 第 4 步：工作流配置和数据准备

这一阶段会生成 notebook 专用运行时配置快照，并把 ENABLE_CALIBRATION / ENABLE_EVALUATE 写回 workflow 配置段。

In [ ]:
RUNTIME_CFG_DIR = CEG_WM_ROOT / 'outputs' / '_notebook_runtime'
RUNTIME_CFG_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG_PATH = RUNTIME_CFG_DIR / f'{RUN_TAG}_paper_ablation_cuda.yaml'
RUN_OUTPUT_ROOT = (CEG_WM_ROOT / OUTPUT_ROOT / RUN_TAG).resolve()

base_cfg = yaml.safe_load((CEG_WM_ROOT / CONFIG_PATH).read_text(encoding='utf-8'))
workflow_cfg = base_cfg['paper_ablation_workflow']
workflow_cfg['output_root'] = OUTPUT_ROOT
workflow_cfg['run_tag'] = RUN_TAG
workflow_cfg['detect_rerun']['enable_calibration'] = ENABLE_CALIBRATION
workflow_cfg['detect_rerun']['enable_evaluate'] = ENABLE_EVALUATE

RUNTIME_CONFIG_PATH.write_text(yaml.safe_dump(base_cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')

print('=' * 80)
print('工作流配置和数据准备')
print('=' * 80)
print(f'CONFIG_PATH={CONFIG_PATH}')
print(f'RUNTIME_CONFIG_PATH={RUNTIME_CONFIG_PATH}')
print(f'RUN_OUTPUT_ROOT={RUN_OUTPUT_ROOT}')
print(f'SELECTED_VARIANTS={SELECTED_VARIANTS}')
print(f'ENABLE_CALIBRATION={ENABLE_CALIBRATION}')
print(f'ENABLE_EVALUATE={ENABLE_EVALUATE}')

## 第 5 步：准备 attestation 环境变量

paper_ablation_cuda 继承 attestation 正式链要求，因此此步会准备 CEG_WM_K_MASTER / CEG_WM_K_PROMPT / CEG_WM_K_SEED。

In [ ]:
import re
import secrets

ATTESTATION_ENV_SPECS = {
    'CEG_WM_K_MASTER': 64,
    'CEG_WM_K_PROMPT': 32,
    'CEG_WM_K_SEED': 32,
}
USER_ATTESTATION_VALUES = {
    'CEG_WM_K_MASTER': '',
    'CEG_WM_K_PROMPT': '',
    'CEG_WM_K_SEED': '',
}
AUTO_GENERATE_MISSING_ATTESTATION_KEYS = True
ATTESTATION_INFO_PATH = WORK_DIR / 'paper_ablation_attestation_env.json'

def _is_valid_hex_secret(value: str, expected_length: int) -> bool:
    if not isinstance(value, str):
        return False
    candidate = value.strip().lower()
    return len(candidate) == expected_length and re.fullmatch(r'[0-9a-f]+', candidate) is not None

def _mask_secret(value: str) -> str:
    return '*' * len(value) if len(value) <= 8 else f'{value[:4]}...{value[-4:]}'

resolved_values = {}
generated_env_vars = []
missing_env_vars = []

for env_name, expected_length in ATTESTATION_ENV_SPECS.items():
    manual_value = USER_ATTESTATION_VALUES.get(env_name, '')
    existing_value = os.environ.get(env_name, '')
    if _is_valid_hex_secret(manual_value, expected_length):
        resolved_value = manual_value.strip().lower()
        source = 'manual_input'
    elif _is_valid_hex_secret(existing_value, expected_length):
        resolved_value = existing_value.strip().lower()
        source = 'existing_env'
    elif AUTO_GENERATE_MISSING_ATTESTATION_KEYS:
        resolved_value = secrets.token_hex(expected_length // 2)
        source = 'generated_for_session'
        generated_env_vars.append(env_name)
    else:
        resolved_value = ''
        source = 'missing'

    if resolved_value:
        os.environ[env_name] = resolved_value
        resolved_values[env_name] = {
            'length': len(resolved_value),
            'masked_value': _mask_secret(resolved_value),
            'source': source,
        }
    else:
        missing_env_vars.append(env_name)

for env_name in ATTESTATION_ENV_SPECS:
    print(f"{env_name}: {resolved_values.get(env_name, {'masked_value': '<absent>', 'source': 'missing'})}")

if missing_env_vars:
    raise RuntimeError(f'attestation 环境变量未就绪：{missing_env_vars}')

ATTESTATION_INFO_PATH.write_text(
    json.dumps({
        'saved_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'generated_env_vars': generated_env_vars,
        'resolved_values': resolved_values,
    }, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(f'✓ attestation 环境信息已保存：{ATTESTATION_INFO_PATH}')

## 第 6 步：运行前预检

在真正执行 ablation workflow 前，检查 GPU、关键脚本、关键模块、配置文件和 InSPyReNet 权重。

In [ ]:
import socket

precheck_results = []

def _record_check(name, ok, detail):
    precheck_results.append({'name': name, 'ok': ok, 'detail': detail})
    print(f"{'✓' if ok else '✗'} {name}: {detail}")

try:
    import torch
    cuda_ok = bool(torch.cuda.is_available())
    _record_check('CUDA 可用', cuda_ok, torch.cuda.get_device_name(0) if cuda_ok else '未检测到 CUDA GPU')
except Exception as exc:
    _record_check('CUDA 可用', False, f'{type(exc).__name__}: {exc}')

required_paths = [
    CEG_WM_ROOT / CONFIG_PATH,
    CEG_WM_ROOT / 'scripts' / 'run_paper_ablation_workflow.py',
    CEG_WM_ROOT / 'main' / 'cli' / 'run_embed.py',
    CEG_WM_ROOT / 'main' / 'cli' / 'run_detect.py',
    CEG_WM_ROOT / 'models' / 'inspyrenet' / 'ckpt_base.pth',
]
for path in required_paths:
    _record_check(f'文件存在: {path.name}', path.exists(), str(path))

required_modules = ['yaml', 'huggingface_hub', 'diffusers', 'transformers', 'safetensors']
for module_name in required_modules:
    try:
        __import__(module_name)
        _record_check(f'模块可导入: {module_name}', True, 'ok')
    except Exception as exc:
        _record_check(f'模块可导入: {module_name}', False, f'{type(exc).__name__}: {exc}')

for env_name, expected_length in ATTESTATION_ENV_SPECS.items():
    value = os.environ.get(env_name, '').strip().lower()
    ok = len(value) == expected_length and all(ch in '0123456789abcdef' for ch in value)
    _record_check(f'attestation 环境变量: {env_name}', ok, f'len={len(value)}' if value else '<absent>')

try:
    from huggingface_hub import HfApi
    socket.setdefaulttimeout(15)
    HfApi().model_info('stabilityai/stable-diffusion-3.5-medium')
    _record_check('HF 模型可访问', True, 'stabilityai/stable-diffusion-3.5-medium')
except Exception as exc:
    _record_check('HF 模型可访问', False, f'{type(exc).__name__}: {exc}')

hard_fail = [item for item in precheck_results if not item['ok'] and item['name'] in {
    'CUDA 可用',
    '文件存在: paper_ablation_cuda.yaml',
    '文件存在: run_paper_ablation_workflow.py',
    '文件存在: run_embed.py',
    '文件存在: run_detect.py',
    '文件存在: ckpt_base.pth',
    '模块可导入: diffusers',
    '模块可导入: transformers',
    'attestation 环境变量: CEG_WM_K_MASTER',
    'attestation 环境变量: CEG_WM_K_PROMPT',
    'attestation 环境变量: CEG_WM_K_SEED',
}]

if hard_fail:
    raise RuntimeError(f'运行前预检未通过，硬性失败项：{hard_fail}')
else:
    print('✓ 运行前预检通过，可以执行 ablation workflow')

## 第 7 步：运行 paper ablation workflow

脚本会执行：
- base embed 一次；
- 多个 detect variant rerun；
- compare summary / compare table 生成。

In [ ]:
command = [
    sys.executable,
    str(CEG_WM_ROOT / 'scripts' / 'run_paper_ablation_workflow.py'),
    '--config', str(RUNTIME_CONFIG_PATH),
    '--run-root', str(RUN_OUTPUT_ROOT),
]
for variant_name in SELECTED_VARIANTS:
    command.extend(['--variant', variant_name])

print('执行命令：')
print(' '.join(command))
run_checked(command, cwd=CEG_WM_ROOT)
print('✓ Paper ablation workflow 执行完成')

## 第 8 步：结果摘要与打包保存到 Google Drive

本阶段读取 compare summary，并把 run_root 打包复制到 Google Drive。

In [ ]:
import zipfile

summary_path = RUN_OUTPUT_ROOT / 'compare' / 'ablation_compare_summary.json'
manifest_path = RUN_OUTPUT_ROOT / 'compare' / 'ablation_manifest.json'
table_path = RUN_OUTPUT_ROOT / 'compare' / 'ablation_compare_table.csv'

summary_obj = json.loads(summary_path.read_text(encoding='utf-8'))
manifest_obj = json.loads(manifest_path.read_text(encoding='utf-8'))

print('=' * 80)
print('Ablation Summary')
print('=' * 80)
print(f"Base Embed Run Root: {summary_obj['base_embed_run_root']}")
print(f"Base Embed Record: {summary_obj['base_embed_record_path']}")
print(f"Manifest: {manifest_path}")
print(f"Compare Table: {table_path}")

for variant_row in summary_obj['variants']:
    print('-' * 80)
    print(f"Variant: {variant_row['variant_name']}")
    print(f"  Attestation Status: {variant_row['attestation_status']}")
    print(f"  Content Attestation Score: {variant_row['content_attestation_score']}")
    print(f"  Event Attestation Score: {variant_row['event_attestation_score']}")
    print(f"  LF/HF/GEO: {variant_row['lf_score']} / {variant_row['hf_score']} / {variant_row['geo_score']}")
    print(f"  Active Score Source: {variant_row['active_score_source']}")
    print(f"  Detect Record: {variant_row['detect_record_path']}")
    print(f"  Input Record: {variant_row['input_record_path']}")
    print(f"  Overrides: {variant_row['variant_overrides']}")

archive_path = WORK_DIR / f'{RUN_TAG}_paper_ablation_outputs.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in RUN_OUTPUT_ROOT.rglob('*'):
        if file_path.is_file():
            archive.write(file_path, arcname=str(file_path.relative_to(RUN_OUTPUT_ROOT.parent)))
print(f'✓ 已生成压缩包：{archive_path}')

if GOOGLE_DRIVE_EXPORT_DIR is not None:
    drive_archive_path = GOOGLE_DRIVE_EXPORT_DIR / archive_path.name
    shutil.copy2(archive_path, drive_archive_path)
    print(f'✓ 压缩包已复制到 Google Drive：{drive_archive_path}')

    drive_summary_path = GOOGLE_DRIVE_EXPORT_DIR / f'{RUN_TAG}_ablation_compare_summary.json'
    drive_manifest_path = GOOGLE_DRIVE_EXPORT_DIR / f'{RUN_TAG}_ablation_manifest.json'
    drive_table_path = GOOGLE_DRIVE_EXPORT_DIR / f'{RUN_TAG}_ablation_compare_table.csv'
    shutil.copy2(summary_path, drive_summary_path)
    shutil.copy2(manifest_path, drive_manifest_path)
    shutil.copy2(table_path, drive_table_path)
    print(f'✓ Compare summary 已复制到 Google Drive：{drive_summary_path}')
    print(f'✓ Manifest 已复制到 Google Drive：{drive_manifest_path}')
    print(f'✓ Compare table 已复制到 Google Drive：{drive_table_path}')

if IN_COLAB:
    try:
        colab_files.download(str(archive_path))
    except Exception as exc:
        print(f'⚠ 浏览器下载失败：{type(exc).__name__}: {exc}')

print('Manifest 中的 variant 列表：')
for variant_entry in manifest_obj['variants']:
    print(f"  - {variant_entry['suffix']}: {variant_entry['variant_run_root']}")